In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Load SCD2 Work Table
# ============================================================

scd2_work_df = spark.table(
    "customer360_dev.control.customer_scd2_work"
)

target_table = "customer360_dev.gold.dim_customer"

In [0]:
new_versions_df = (
    scd2_work_df
    .filter(
        F.col("scd2_action").isin(
            "NEW_CUSTOMER",
            "TYPE2_CHANGE"
        )
    )
)

new_version_count = new_versions_df.count()

print(
    "New SCD2 versions to insert:",
    new_version_count
)

New SCD2 versions to insert: 0


In [0]:
new_versions_df = (
    new_versions_df

    .withColumn(
        "effective_from",
        F.coalesce(
            F.col("source_updated_at"),
            F.current_timestamp()
        )
    )

    .withColumn(
        "effective_to",
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )

    .withColumn(
        "is_current",
        F.lit(True)
    )

    .withColumn(
        "_scd_created_at",
        F.current_timestamp()
    )

    .withColumn(
        "_scd_updated_at",
        F.current_timestamp()
    )
)

In [0]:
insert_df = (
    new_versions_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone_number",
        "date_of_birth",
        "gender",
        "signup_date",
        "customer_status",
        "loyalty_tier",
        "preferred_language",

        "effective_from",
        "effective_to",
        "is_current",

        "source_updated_at"
    )
)

In [0]:
if new_version_count > 0:

    (
        insert_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            target_table
        )
    )

    print(
        "New Customer SCD2 versions inserted successfully."
    )

else:

    print(
        "No NEW_CUSTOMER or TYPE2_CHANGE rows found."
    )

No NEW_CUSTOMER or TYPE2_CHANGE rows found.


In [0]:
current_customer_count = (
    spark.table(target_table)
    .filter(
        F.col("is_current") == True
    )
    .select("customer_id")
    .distinct()
    .count()
)

print(
    "Customers with current Gold version:",
    current_customer_count
)

Customers with current Gold version: 986000


In [0]:
multiple_current_count = (
    spark.table(target_table)
    .filter(
        F.col("is_current") == True
    )
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    "Customers with multiple current versions:",
    multiple_current_count
)

Customers with multiple current versions: 0


In [0]:
display(
    spark.table(target_table)
    .orderBy(
        "customer_id",
        "effective_from"
    )
    .select(
        "customer_sk",
        "customer_id",
        "loyalty_tier",
        "customer_status",
        "effective_from",
        "effective_to",
        "is_current",
        "source_updated_at"
    )
    .limit(100)
)

customer_sk,customer_id,loyalty_tier,customer_status,effective_from,effective_to,is_current,source_updated_at
1,CUST000000001,BRONZE,ACTIVE,2024-02-16T12:07:00.000Z,9999-12-31T23:59:59.000Z,true,2024-02-16T12:07:00.000Z
2,CUST000000002,SILVER,ACTIVE,2024-04-02T15:14:00.000Z,9999-12-31T23:59:59.000Z,true,2024-04-02T15:14:00.000Z
3,CUST000000003,SILVER,ACTIVE,2024-05-18T18:21:00.000Z,9999-12-31T23:59:59.000Z,true,2024-05-18T18:21:00.000Z
924734,CUST000000004,GOLD,INACTIVE,2024-07-03T21:28:00.000Z,9999-12-31T23:59:59.000Z,true,2024-07-03T21:28:00.000Z
4,CUST000000005,PLATINUM,SUSPENDED,2024-08-19T00:35:00.000Z,9999-12-31T23:59:59.000Z,true,2024-08-19T00:35:00.000Z
5,CUST000000006,BRONZE,ACTIVE,2024-10-04T03:42:00.000Z,9999-12-31T23:59:59.000Z,true,2024-10-04T03:42:00.000Z
924735,CUST000000007,BRONZE,ACTIVE,2024-11-19T06:49:00.000Z,9999-12-31T23:59:59.000Z,true,2024-11-19T06:49:00.000Z
6,CUST000000008,SILVER,ACTIVE,2025-01-03T09:56:00.000Z,9999-12-31T23:59:59.000Z,true,2025-01-03T09:56:00.000Z
7,CUST000000009,SILVER,ACTIVE,2025-02-18T12:03:00.000Z,9999-12-31T23:59:59.000Z,true,2025-02-18T12:03:00.000Z
8,CUST000000010,GOLD,INACTIVE,2025-04-05T15:10:00.000Z,9999-12-31T23:59:59.000Z,true,2025-04-05T15:10:00.000Z
